# 15 Command | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: enkapsulacja zadania jako obiektu
2. 👥 Uczestnicy: Command / Invoker / Receiver
3. ↩️ Implementacja z `execute()` i `undo()`
4. 📚 Historia polecen (undo stack)
5. 📬 Kolejka polecen (`queue.Queue`)

## 1. 🔹 Problem: enkapsulacja zadania jako obiektu

Command (Polecenie) to wzorzec behawioralny zamieniajacy
zadanie w samodzielny obiekt zawierajacy wszystkie
informacje potrzebne do jego wykonania.

Problem: jak:
- Parametryzowac obiekty operacjami
- Kolejkowac operacje i wykonywac asynchronicznie
- Realizowac cofanie operacji (undo/redo)
- Logowac operacje

Analogacja: restauracja - kucharz nie komunikuje sie
bezposrednio z kelnerem. Zamowienie (Command) jest
oddawane do kuchni, mozna je anulowac, podzielac,
zakolejkowac.

Kluczowe zalety:
- Oddzielenie zrodla zadania od wykonawcy
- Undo/Redo przez przechowywanie historii
- Makro polecenia (kompozycja polecen)
- Kolejkowanie i asynchroniczne wykonanie

> 💡 Jesli potrzebujesz undo/redo, queueing lub
> transaction logging - to sygnaly ze potrzebujesz Command.

In [ ]:
# Problem bez Command: bezposrednie wywolania
class Light:
    def on(self) -> None: print('Light ON')
    def off(self) -> None: print('Light OFF')

class Thermostat:
    def set_temp(self, temp: float) -> None: print(f'Temp: {temp}')

# Bez Command: Pilot musi znac wszystkie urzadzenia
class RemoteControl:
    def __init__(self, light: Light, thermostat: Thermostat):
        self._light = light
        self._thermostat = thermostat

    def press_button_1(self): self._light.on()
    def press_button_2(self): self._light.off()
    def press_button_3(self): self._thermostat.set_temp(21.0)
    # Dodanie nowego urzadzenia = zmiana RemoteControl!

problems = [
    'RemoteControl zna Light i Thermostat bezposrednio',
    'Nie mozna dodac undo - nie wiemy co bylo wczesniej',
    'Nie mozna kolejkowac - natychmiastowe wykonanie',
    'Dodanie nowego urzadzenia wymaga zmiany Remote',
]
for p in problems: print(f'- {p}')

---

### 🐍 Cwiczenia - problem

1. Rozszerz `RemoteControl` o obsługe `Fan` (wlacz/wylacz, predkosc).
   Policz ile klas/metod musisz zmienic.
2. Jak dodalbys undo do `RemoteControl` bez wzorca Command?
   Opisz problemy tego podejscia.
3. *(Trudniejsze)* Ile klas musisz zmienic aby kolejkowac
   operacje i wykonywac je po 5 sekundach?

In [ ]:
# Cwiczenie 1: koszt rozszerzenia
class Fan:
    def on(self) -> None: print('Fan ON')
    def off(self) -> None: print('Fan OFF')
    def set_speed(self, speed: int) -> None: print(f'Fan speed: {speed}')

print('Bez Command - zeby dodac Fan do Remote:')
changes = [
    'RemoteControl.__init__: dodac self._fan = fan',
    'RemoteControl.press_button_4: self._fan.on()',
    'RemoteControl.press_button_5: self._fan.off()',
    'RemoteControl.press_button_6: self._fan.set_speed(3)',
]
for c in changes: print(f'  - {c}')
print(f'Lacznie zmian: {len(changes)}')
print('Z Command: 0 zmian w RemoteControl!')

In [ ]:
# Cwiczenie 2: undo bez Command
class RemoteWithUndoBad:
    def __init__(self, light: Light):
        self._light = light
        self._last_action = None  # musimy pamietac
    def press_on(self): self._light.on(); self._last_action = 'on'
    def press_off(self): self._light.off(); self._last_action = 'off'
    def undo(self):
        # Problem: duze if-else, nie skaluje
        if self._last_action == 'on': self._light.off()
        elif self._last_action == 'off': self._light.on()

problems = [
    'Musisz przechowywac informacje o ostatniej akcji',
    'Nie mozna undo dla wielu operacji (tylko 1 poziom)',
    'If-else rosnie z kazdym nowym poleceniem',
    'Nie mozna undo dla operacji ze stanem (np. temp: ile bylo poprzednio?)',
]
for p in problems: print(f'- {p}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: kolejkowanie bez Command
print('Kolejkowanie bez Command - trudnosci:')
difficulties = [
    '1. Musisz przechowywac metody i argumenty osobno',
    '2. Nie ma wspolnego interfejsu dla wszystkich operacji',
    '3. Trudno serializowac (np. do bazy) bez Command obiektu',
    '4. Tymczasowe opoznienie wymaga zlozonej logiki',
]
for d in difficulties: print(f'  {d}')
print()
print('Z Command: queue.put(command), worker.execute(queue.get())')

## 2. 🔹 Uczestnicy: Command / Invoker / Receiver

Uczestnicy wzorca Command:

**Command** (interfejs):
- `execute()` - wykonaj polecenie
- `undo()` - cofnij polecenie (opcjonalne)

**ConcreteCommand**:
- Implementuje Command
- Trzyma referencje do Receiver
- Przechowuje stan potrzebny do undo

**Receiver**:
- Obiekt ktory faktycznie wykonuje akcje
- Nie wie o Command

**Invoker**:
- Wykonuje Command przez `execute()`
- Nie wie jak polecenie jest zaimplementowane
- Moze przechowywac historie Command

**Client**:
- Tworzy ConcreteCommand i laczy z Receiver
- Przekazuje Command do Invoker

Analogie:
- Receiver = kucharz, Command = zamowienie,
  Invoker = kelner, Client = klient restauracji

In [ ]:
from abc import ABC, abstractmethod

# Command interface
class Command(ABC):
    @abstractmethod
    def execute(self) -> None: ...
    @abstractmethod
    def undo(self) -> None: ...

# Receiver
class SmartLight:
    def __init__(self, name: str):
        self.name = name
        self._brightness = 0
        self._is_on = False

    def on(self) -> None: self._is_on = True; print(f'{self.name}: ON (brightness={self._brightness})')
    def off(self) -> None: self._is_on = False; print(f'{self.name}: OFF')
    def set_brightness(self, level: int) -> None: self._brightness = level; print(f'{self.name}: brightness={level}')
    def get_brightness(self) -> int: return self._brightness
    def is_on(self) -> bool: return self._is_on

# ConcreteCommands
class TurnOnCommand(Command):
    def __init__(self, light: SmartLight): self._light = light
    def execute(self) -> None: self._light.on()
    def undo(self) -> None: self._light.off()

class TurnOffCommand(Command):
    def __init__(self, light: SmartLight): self._light = light
    def execute(self) -> None: self._light.off()
    def undo(self) -> None: self._light.on()

class SetBrightnessCommand(Command):
    def __init__(self, light: SmartLight, level: int):
        self._light = light; self._level = level; self._prev = 0
    def execute(self) -> None:
        self._prev = self._light.get_brightness()
        self._light.set_brightness(self._level)
    def undo(self) -> None:
        self._light.set_brightness(self._prev)

# Invoker
class RemoteControl:
    def __init__(self):
        self._slots: dict[str, Command] = {}
        self._history: list[Command] = []

    def set_command(self, button: str, command: Command) -> None:
        self._slots[button] = command

    def press(self, button: str) -> None:
        if button in self._slots:
            self._slots[button].execute()
            self._history.append(self._slots[button])

    def undo_last(self) -> None:
        if self._history:
            self._history.pop().undo()

# Client: laczy wszystko
living_room = SmartLight('Living Room')
bedroom = SmartLight('Bedroom')

remote = RemoteControl()
remote.set_command('1', TurnOnCommand(living_room))
remote.set_command('2', TurnOffCommand(living_room))
remote.set_command('3', SetBrightnessCommand(living_room, 75))
remote.set_command('4', TurnOnCommand(bedroom))

remote.press('1')
remote.press('3')
remote.press('4')
remote.undo_last()
remote.undo_last()

---

### 🐍 Cwiczenia - uczestnicy

1. Dodaj `Thermostat` (Receiver) i `SetTemperatureCommand`.
   Przetestuj bez zmiany `RemoteControl`.
2. Napisz `NullCommand` (pusta implementacja) dla nieprzypisanych
   przyciskow pilota (zamiast if-check).
3. *(Trudniejsze)* Napisz `CompositeCommand(commands: list)`
   wykonujacy wiele polecen jako jedno.

In [ ]:
# Cwiczenie 1: Thermostat
class Thermostat:
    def __init__(self): self._temp = 20.0
    def set_temp(self, temp: float) -> None: self._temp = temp; print(f'Thermostat: {temp}C')
    def get_temp(self) -> float: return self._temp

class SetTempCommand(Command):
    def __init__(self, thermostat: Thermostat, temp: float):
        self._t = thermostat; self._temp = temp; self._prev = 20.0
    def execute(self) -> None: self._prev = self._t.get_temp(); self._t.set_temp(self._temp)
    def undo(self) -> None: self._t.set_temp(self._prev)

thermostat = Thermostat()
remote.set_command('5', SetTempCommand(thermostat, 22.0))
remote.press('5')
remote.undo_last()

In [ ]:
# Cwiczenie 2: NullCommand
class NullCommand(Command):
    def execute(self) -> None: pass  # nic nie robi
    def undo(self) -> None: pass

# Inicjalizuj remote z NullCommand zamiast sprawdzania None
null_remote = RemoteControl()
for btn in ['A', 'B', 'C', 'D']:
    null_remote.set_command(btn, NullCommand())

null_remote.set_command('A', TurnOnCommand(living_room))
null_remote.press('A')  # dziala
null_remote.press('B')  # NullCommand - brak efektu, brak bledu

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: CompositeCommand
class CompositeCommand(Command):
    def __init__(self, *commands: Command):
        self._commands = commands
    def execute(self) -> None:
        for cmd in self._commands:
            cmd.execute()
    def undo(self) -> None:
        for cmd in reversed(self._commands):
            cmd.undo()

home_theater = CompositeCommand(
    TurnOnCommand(living_room),
    SetBrightnessCommand(living_room, 30),
    SetTempCommand(thermostat, 21.0),
)

remote.set_command('9', home_theater)
print('\n--- Movie mode ---')
remote.press('9')
print('\n--- Undo movie mode ---')
remote.undo_last()

## 3. 🔹 Implementacja z `execute()` i `undo()`

Klucz do poprawnego undo: ConcreteCommand musi
zapamiętac stan PRZED wykonaniem.

Strategia dla undo:
1. W `execute()` zapisz stary stan do pola prywatnego
2. W `undo()` przywroc zapisany stan

```python
class SetTemperatureCommand:
    def __init__(self, device, new_temp):
        self._device = device
        self._new_temp = new_temp
        self._old_temp = None  # zostanie zapisany w execute

    def execute(self):
        self._old_temp = self._device.get_temp()  # zapamietaj
        self._device.set_temp(self._new_temp)

    def undo(self):
        self._device.set_temp(self._old_temp)     # przywroc
```

Wazne: Command musi byc idempotentny przy wielokrotnym
execute - lub musimy to obsluzyc (np. przez flagę).

In [ ]:
from abc import ABC, abstractmethod

# Edytor tekstu z pelna historia
class TextEditor:
    def __init__(self) -> None: self._text = ''; self._selection = (0, 0)

    @property
    def text(self) -> str: return self._text

    def insert(self, pos: int, text: str) -> None:
        self._text = self._text[:pos] + text + self._text[pos:]

    def delete(self, pos: int, length: int) -> str:
        removed = self._text[pos:pos+length]
        self._text = self._text[:pos] + self._text[pos+length:]
        return removed

    def replace(self, pos: int, length: int, new_text: str) -> str:
        removed = self._text[pos:pos+length]
        self._text = self._text[:pos] + new_text + self._text[pos+length:]
        return removed

class TextCommand(ABC):
    @abstractmethod
    def execute(self) -> None: ...
    @abstractmethod
    def undo(self) -> None: ...
    @abstractmethod
    def description(self) -> str: ...

class InsertTextCommand(TextCommand):
    def __init__(self, editor: TextEditor, pos: int, text: str):
        self._e = editor; self._pos = pos; self._text = text
    def execute(self) -> None: self._e.insert(self._pos, self._text)
    def undo(self) -> None: self._e.delete(self._pos, len(self._text))
    def description(self) -> str: return f'Insert {self._text!r} at {self._pos}'

class DeleteTextCommand(TextCommand):
    def __init__(self, editor: TextEditor, pos: int, length: int):
        self._e = editor; self._pos = pos; self._len = length; self._deleted = ''
    def execute(self) -> None: self._deleted = self._e.delete(self._pos, self._len)
    def undo(self) -> None: self._e.insert(self._pos, self._deleted)
    def description(self) -> str: return f'Delete {self._len} chars at {self._pos}'

class ReplaceTextCommand(TextCommand):
    def __init__(self, editor: TextEditor, pos: int, length: int, new_text: str):
        self._e = editor; self._pos = pos; self._len = length
        self._new = new_text; self._old = ''
    def execute(self) -> None: self._old = self._e.replace(self._pos, self._len, self._new)
    def undo(self) -> None: self._e.replace(self._pos, len(self._new), self._old)
    def description(self) -> str: return f'Replace at {self._pos}: {self._old!r} -> {self._new!r}'

editor = TextEditor()
commands = [
    InsertTextCommand(editor, 0, 'Hello'),
    InsertTextCommand(editor, 5, ' World'),
    ReplaceTextCommand(editor, 6, 5, 'Python'),
    InsertTextCommand(editor, 12, '!'),
]

for cmd in commands:
    cmd.execute()
    print(f'{cmd.description()}: "{editor.text}"')

print('\n--- Undo step by step ---')
for cmd in reversed(commands):
    cmd.undo()
    print(f'Undo {cmd.description()}: "{editor.text}"')

---

### 🐍 Cwiczenia - execute / undo

1. Napisz `CopyTextCommand` i `PasteTextCommand` dla edytora.
   Schowek (clipboard) niech bedzie parametrem konstruktora.
2. Napisz `DrawCommand` dla obrazka: `canvas.draw_pixel(x, y, color)`.
   `undo()` przywraca stary kolor piksela.
3. *(Trudniejsze)* Napisz `BatchCommand(commands)` ktory
   w razie bledu jednego polecenia cofa wszystkie poprzednie.

In [ ]:
# Cwiczenie 1: CopyTextCommand i PasteTextCommand
clipboard = {'text': ''}

class CopyTextCommand(TextCommand):
    def __init__(self, editor: TextEditor, start: int, end: int, cb: dict):
        self._e = editor; self._start = start; self._end = end; self._cb = cb
        self._prev_cb = ''
    def execute(self) -> None:
        self._prev_cb = self._cb.get('text', '')
        self._cb['text'] = self._e.text[self._start:self._end]
        print(f'Copied: {self._cb["text"]!r}')
    def undo(self) -> None: self._cb['text'] = self._prev_cb
    def description(self) -> str: return f'Copy [{self._start}:{self._end}]'

class PasteTextCommand(TextCommand):
    def __init__(self, editor: TextEditor, pos: int, cb: dict):
        self._e = editor; self._pos = pos; self._cb = cb; self._len = 0
    def execute(self) -> None:
        text = self._cb.get('text', '')
        self._len = len(text)
        self._e.insert(self._pos, text)
    def undo(self) -> None: self._e.delete(self._pos, self._len)
    def description(self) -> str: return f'Paste at {self._pos}'

editor2 = TextEditor()
InsertTextCommand(editor2, 0, 'Hello World').execute()
print(editor2.text)
copy_cmd = CopyTextCommand(editor2, 6, 11, clipboard)
copy_cmd.execute()
paste_cmd = PasteTextCommand(editor2, 0, clipboard)
paste_cmd.execute()
print(editor2.text)  # WorldHello World
paste_cmd.undo()
print(editor2.text)  # Hello World

In [ ]:
# Cwiczenie 2: DrawCommand
class Canvas:
    def __init__(self, w: int, h: int):
        self._pixels = [[0] * w for _ in range(h)]
    def draw_pixel(self, x: int, y: int, color: int) -> None:
        self._pixels[y][x] = color
    def get_pixel(self, x: int, y: int) -> int:
        return self._pixels[y][x]
    def display(self) -> None:
        for row in self._pixels:
            print(' '.join(str(p) for p in row))

class DrawPixelCommand(Command):
    def __init__(self, canvas: Canvas, x: int, y: int, color: int):
        self._canvas = canvas; self._x = x; self._y = y; self._color = color
        self._prev_color = 0
    def execute(self) -> None:
        self._prev_color = self._canvas.get_pixel(self._x, self._y)
        self._canvas.draw_pixel(self._x, self._y, self._color)
    def undo(self) -> None:
        self._canvas.draw_pixel(self._x, self._y, self._prev_color)

canvas = Canvas(3, 3)
cmds = [
    DrawPixelCommand(canvas, 0, 0, 1),
    DrawPixelCommand(canvas, 1, 1, 2),
    DrawPixelCommand(canvas, 2, 2, 3),
]
for cmd in cmds: cmd.execute()
print('After drawing:')
canvas.display()
for cmd in reversed(cmds): cmd.undo()
print('After undo:')
canvas.display()

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: BatchCommand z rollback
class BatchCommand(Command):
    def __init__(self, *commands: Command):
        self._commands = commands
        self._executed: list[Command] = []

    def execute(self) -> None:
        self._executed.clear()
        try:
            for cmd in self._commands:
                cmd.execute()
                self._executed.append(cmd)
        except Exception as e:
            print(f'Error: {e}. Rolling back...')
            for cmd in reversed(self._executed):
                cmd.undo()
            self._executed.clear()
            raise

    def undo(self) -> None:
        for cmd in reversed(self._executed):
            cmd.undo()

class FailingCommand(Command):
    def execute(self) -> None: raise RuntimeError('Simulated failure')
    def undo(self) -> None: pass

editor3 = TextEditor()
batch = BatchCommand(
    InsertTextCommand(editor3, 0, 'Hello'),
    InsertTextCommand(editor3, 5, ' World'),
    FailingCommand(),  # powinno triggerowac rollback
)
try:
    batch.execute()
except RuntimeError:
    pass
print(f'Text after failed batch: "{editor3.text}"')  # powinno byc puste

## 4. 🔹 Historia polecen (undo stack)

Historia polecen (Command History) umozliwia undo/redo:

**Stos undo (undo stack)**:
- `history: list[Command]` - wykonane polecenia
- `execute(cmd)`: `cmd.execute()` + `history.append(cmd)`
- `undo()`: `cmd = history.pop()`, `cmd.undo()`

**Stos redo (redo stack)**:
- Gdy cofniemy polecenie, idzie na redo_stack
- `redo()`: `cmd = redo_stack.pop()`, `cmd.execute()` + `history.append(cmd)`
- WAZNE: nowe execute() CZYSCI redo_stack!

Granice historii:
- `max_history_size` - limit rozmiary stosu
- Gdy przekroczony: usuwamy najstarsze polecenie
- Lub: blokujemy execute gdy za duzo polecen

Transakcje:
- Kilka polecen jako jedna jednostka undo
- `begin_transaction()` / `commit()` / `rollback()`

In [ ]:
from collections import deque

class CommandHistory:
    def __init__(self, max_size: int = 100) -> None:
        self._history: deque[Command] = deque(maxlen=max_size)
        self._redo_stack: list[Command] = []

    def execute(self, command: Command) -> None:
        command.execute()
        self._history.append(command)
        self._redo_stack.clear()  # nowe execute kasuje redo

    def undo(self) -> bool:
        if not self._history:
            print('Nothing to undo')
            return False
        cmd = self._history.pop()
        cmd.undo()
        self._redo_stack.append(cmd)
        return True

    def redo(self) -> bool:
        if not self._redo_stack:
            print('Nothing to redo')
            return False
        cmd = self._redo_stack.pop()
        cmd.execute()
        self._history.append(cmd)
        return True

    def undo_all(self) -> None:
        while self._history:
            self.undo()

    @property
    def history_size(self) -> int: return len(self._history)

    @property
    def redo_size(self) -> int: return len(self._redo_stack)

# Pelny przyklad: kalkulator z undo/redo
class Calculator:
    def __init__(self): self._value = 0.0
    def add(self, n: float) -> None: self._value += n
    def sub(self, n: float) -> None: self._value -= n
    def mul(self, n: float) -> None: self._value *= n
    @property
    def value(self) -> float: return self._value

class AddCommand(Command):
    def __init__(self, calc: Calculator, n: float): self._c = calc; self._n = n
    def execute(self) -> None: self._c.add(self._n)
    def undo(self) -> None: self._c.sub(self._n)
    def description(self) -> str: return f'+{self._n}'

class MulCommand(Command):
    def __init__(self, calc: Calculator, n: float): self._c = calc; self._n = n
    def execute(self) -> None: self._c.mul(self._n)
    def undo(self) -> None: self._c.mul(1 / self._n)
    def description(self) -> str: return f'*{self._n}'

calc = Calculator()
history = CommandHistory(max_size=10)

for cmd in [
    AddCommand(calc, 10),
    MulCommand(calc, 3),
    AddCommand(calc, 5),
]:
    history.execute(cmd)
    print(f'{cmd.description()}: {calc.value}')

print(f'History: {history.history_size}, Redo: {history.redo_size}')
history.undo()
print(f'After undo: {calc.value}')
history.undo()
print(f'After undo: {calc.value}')
history.redo()
print(f'After redo: {calc.value}')
history.execute(AddCommand(calc, 100))  # clears redo
print(f'New cmd: {calc.value}')
history.redo()  # nothing - redo cleared
history.undo_all()
print(f'After undo_all: {calc.value}')

---

### 🐍 Cwiczenia - historia polecen

1. Dodaj do `CommandHistory` metode `get_history() -> list[str]`
   zwracajaca opisy polecen w kolejnosci wykonania.
2. Napisz edytor tekstu korzystajacy z `CommandHistory` z
   pelnym undo/redo. Przetestuj 5 operacji.
3. *(Trudniejsze)* Zaimplementuj transakcje: `begin()`, `commit()`,
   `rollback()` - wiele polecen jako jedna jednostka undo.

In [ ]:
# Cwiczenie 1: get_history
class CommandHistoryV2(CommandHistory):
    def get_history(self) -> list[str]:
        result = []
        for cmd in self._history:
            if hasattr(cmd, 'description'):
                result.append(cmd.description())
            else:
                result.append(type(cmd).__name__)
        return result

calc2 = Calculator()
history2 = CommandHistoryV2()
for cmd in [AddCommand(calc2, 10), MulCommand(calc2, 2), AddCommand(calc2, 5)]:
    history2.execute(cmd)

print('History:', history2.get_history())
print('Value:', calc2.value)

In [ ]:
# Cwiczenie 2: edytor z CommandHistory
editor4 = TextEditor()
history4 = CommandHistoryV2()

ops = [
    InsertTextCommand(editor4, 0, 'Hello'),
    InsertTextCommand(editor4, 5, ' World'),
    ReplaceTextCommand(editor4, 0, 5, 'Goodbye'),
    InsertTextCommand(editor4, 12, '!'),
    DeleteTextCommand(editor4, 8, 5),
]
for op in ops:
    history4.execute(op)
    print(f'{op.description()!r:40}: {editor4.text!r}')

print('\nUndo 3 razy:')
for _ in range(3): history4.undo(); print(f'  {editor4.text!r}')
print('\nRedo 2 razy:')
for _ in range(2): history4.redo(); print(f'  {editor4.text!r}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: transakcje
class TransactionalHistory(CommandHistoryV2):
    def __init__(self):
        super().__init__()
        self._transaction: list[Command] | None = None

    def begin(self) -> None:
        self._transaction = []
        print('Transaction started')

    def execute(self, command: Command) -> None:
        command.execute()
        if self._transaction is not None:
            self._transaction.append(command)
        else:
            self._history.append(command)
            self._redo_stack.clear()

    def commit(self) -> None:
        if self._transaction is None: return
        # Zapakuj transakcje jako jeden BatchCommand
        batch = CompositeCommand(*self._transaction)
        self._history.append(batch)
        self._redo_stack.clear()
        self._transaction = None
        print('Transaction committed')

    def rollback(self) -> None:
        if self._transaction is None: return
        for cmd in reversed(self._transaction):
            cmd.undo()
        self._transaction = None
        print('Transaction rolled back')

editor5 = TextEditor()
tx_history = TransactionalHistory()

# Udana transakcja
tx_history.begin()
tx_history.execute(InsertTextCommand(editor5, 0, 'Hello'))
tx_history.execute(InsertTextCommand(editor5, 5, ' World'))
tx_history.commit()
print(f'After commit: {editor5.text!r}')

# Nieudana transakcja
tx_history.begin()
tx_history.execute(InsertTextCommand(editor5, 11, '!'))
tx_history.execute(InsertTextCommand(editor5, 12, '?'))
tx_history.rollback()
print(f'After rollback: {editor5.text!r}')

tx_history.undo()  # cofa cala zatwierdzona transakcje
print(f'After undo: {editor5.text!r}')

## 5. 🔹 Kolejka polecen (`queue.Queue`)

Kolejka polecen (Command Queue) umozliwia:
- Asynchroniczne wykonanie polecen
- Planowanie polecen na pozniej
- Wykonanie polecen przez osobny watek/proces
- Throttling (max N polecen na sekunde)

`queue.Queue` - thread-safe kolejka FIFO:
- `queue.put(command)` - dodaje do kolejki
- `queue.get()` - pobiera z kolejki (blokuje jesli pusta)
- `queue.task_done()` - sygnalizuje zakonczenie

Worker pattern:
```python
def worker(q: queue.Queue):
    while True:
        cmd = q.get()
        if cmd is None: break  # poison pill
        cmd.execute()
        q.task_done()
```

Command jako zadanie (Task):
- `ScheduledCommand(command, delay)`
- `RetryCommand(command, max_retries)`
- `LoggingCommand(command, logger)`

In [ ]:
import queue
import threading
import time

class PrintCommand(Command):
    def __init__(self, message: str, delay: float = 0):
        self._msg = message; self._delay = delay
    def execute(self) -> None:
        if self._delay: time.sleep(self._delay)
        print(f'[{time.strftime("%H:%M:%S")}] {self._msg}')
    def undo(self) -> None: print(f'[Undo] {self._msg}')

# Synchroniczna kolejka
class CommandQueue:
    def __init__(self) -> None:
        self._queue: list[Command] = []

    def add(self, command: Command) -> None:
        self._queue.append(command)

    def run_all(self) -> None:
        while self._queue:
            self._queue.pop(0).execute()

    def run_n(self, n: int) -> None:
        for _ in range(min(n, len(self._queue))):
            self._queue.pop(0).execute()

# Command z dekoracjami
class RetryCommand(Command):
    def __init__(self, command: Command, max_retries: int = 3):
        self._cmd = command; self._max = max_retries
    def execute(self) -> None:
        for attempt in range(1, self._max + 1):
            try:
                self._cmd.execute()
                return
            except Exception as e:
                if attempt == self._max: raise
                print(f'Retry {attempt}: {e}')
    def undo(self) -> None: self._cmd.undo()

class LoggingCommand(Command):
    def __init__(self, command: Command):
        self._cmd = command
    def execute(self) -> None:
        print(f'[LOG] Executing: {type(self._cmd).__name__}')
        self._cmd.execute()
        print(f'[LOG] Done: {type(self._cmd).__name__}')
    def undo(self) -> None:
        print(f'[LOG] Undoing: {type(self._cmd).__name__}')
        self._cmd.undo()

# Uzycie kolejki
cmd_queue = CommandQueue()
cmd_queue.add(PrintCommand('Task 1'))
cmd_queue.add(LoggingCommand(PrintCommand('Task 2')))
cmd_queue.add(PrintCommand('Task 3'))
print('Running all commands:')
cmd_queue.run_all()

---

### 🐍 Cwiczenia - kolejka polecen

1. Napisz `PriorityCommandQueue` gdzie polecenia maja priorytet
   (1-10). Wykonuj najpierw polecenia z wyzszym priorytetem.
2. Napisz `DelayedCommand(command, delay_ms)` wykonujacy sie
   dopiero po podanym opoznieniu.
3. *(Trudniejsze)* Napisz `WorkflowEngine(steps: list[Command])`
   z metodami `run()`, `pause()`, `resume()` i `status()`.

In [ ]:
# Cwiczenie 1: PriorityCommandQueue
import heapq

class PriorityCommandQueue:
    def __init__(self):
        self._heap = []  # (priority, counter, command)
        self._counter = 0

    def add(self, command: Command, priority: int = 5) -> None:
        # hint: wyzszy priorytet = wykonaj wcześniej; uzyj negacji dla max-heap
        heapq.heappush(self._heap, (-priority, self._counter, command))
        self._counter += 1

    def run_all(self) -> None:
        while self._heap:
            _, _, cmd = heapq.heappop(self._heap)
            cmd.execute()

pq = PriorityCommandQueue()
pq.add(PrintCommand('Low priority task'), priority=1)
pq.add(PrintCommand('High priority task'), priority=10)
pq.add(PrintCommand('Medium priority task'), priority=5)
print('Running priority queue:')
pq.run_all()

In [ ]:
# Cwiczenie 2: DelayedCommand
class DelayedCommand(Command):
    def __init__(self, command: Command, delay_ms: float):
        self._cmd = command
        self._delay = delay_ms / 1000

    def execute(self) -> None:
        # hint: time.sleep(self._delay)
        time.sleep(self._delay)
        self._cmd.execute()

    def undo(self) -> None: self._cmd.undo()

start = time.perf_counter()
DelayedCommand(PrintCommand('Delayed!'), delay_ms=100).execute()
print(f'Elapsed: {(time.perf_counter()-start)*1000:.0f}ms')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: WorkflowEngine
class WorkflowEngine:
    def __init__(self, *steps: Command):
        self._steps = list(steps)
        self._current = 0
        self._paused = False

    def run(self) -> None:
        self._paused = False
        while self._current < len(self._steps) and not self._paused:
            step = self._steps[self._current]
            print(f'[{self._current+1}/{len(self._steps)}] ', end='')
            step.execute()
            self._current += 1

    def pause(self) -> None:
        self._paused = True
        print(f'Paused at step {self._current+1}')

    def resume(self) -> None:
        self.run()

    def status(self) -> dict:
        return {
            'total': len(self._steps),
            'completed': self._current,
            'remaining': len(self._steps) - self._current,
            'paused': self._paused,
        }

workflow = WorkflowEngine(
    PrintCommand('Step 1: Initialize'),
    PrintCommand('Step 2: Validate'),
    PrintCommand('Step 3: Process'),
    PrintCommand('Step 4: Store'),
    PrintCommand('Step 5: Notify'),
)

print('Starting workflow:')
workflow.run()
print('\nStatus:', workflow.status())